# ZINC Guidance Bootstrap

Bootstrap a regression guidance predictor from generated ZINC molecules, inspect random samples from adaptive feasibility buckets each cycle, then compare unguided and guided interpolation samples.

In [ ]:
# Enable inline plotting and autoreload, then resolve the repo and NSPPK roots.
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.core.display import HTML
from sklearn.model_selection import train_test_split

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

import sys
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / '_notebook_bootstrap.py').exists():
        _root_str = str(_root.resolve())
        if _root_str not in sys.path:
            sys.path.insert(0, _root_str)
        break
else:
    raise ModuleNotFoundError("Could not locate '_notebook_bootstrap.py' from current notebook directory.")
from _notebook_bootstrap import configure_notebook

globals().update(configure_notebook(require_nsppk=True, print_torch=True))

del _root, _root_str

from conditional_node_field_graph_generator.extensions.demo.pipeline import (
    build_graph_generator,
    fit_graph_generator,
)
from conditional_node_field_graph_generator.extensions.molecular import (
    build_zinc_graph_corpus,
    download_zinc_dataset,
    draw_molecules,
    extract_zinc_targets,
    load_zinc_graph_dataset,
)
from conditional_node_field_graph_generator.persistence import load_graph_generator


In [ ]:
# Notebook-local helpers for summaries and per-cycle bucket plots.


def show_molecules(graphs, n=None, title=None, legends=None, n_graphs_per_line=4):
    graphs = list(graphs)
    if n is not None:
        graphs = graphs[:n]
    if title:
        print(title)
    if not graphs:
        print('No graphs to display.')
        return
    draw_molecules(
        graphs,
        legends=legends,
        n_graphs_per_line=n_graphs_per_line,
    )


def summarize_graphs(graphs, prefix='dataset'):
    node_counts = np.array([graph.number_of_nodes() for graph in graphs], dtype=int)
    edge_counts = np.array([graph.number_of_edges() for graph in graphs], dtype=int)
    print(f'{prefix}: {len(graphs)} graphs')
    print(f'{prefix}: node count min/median/max = {node_counts.min()}/{int(np.median(node_counts))}/{node_counts.max()}')
    print(f'{prefix}: edge count min/median/max = {edge_counts.min()}/{int(np.median(edge_counts))}/{edge_counts.max()}')


def summarize_generated_batch(batch, label):
    if len(batch) == 0:
        return {
            'label': label,
            'count': 0,
            'feasible_rate': 0.0,
            'mean_violations': 0.0,
            'median_violations': 0.0,
            'mean_target': 0.0,
            'median_target': 0.0,
        }
    return {
        'label': label,
        'count': len(batch),
        'feasible_rate': float(np.mean(batch.feasible_mask)),
        'mean_violations': float(np.mean(batch.violation_counts)),
        'median_violations': float(np.median(batch.violation_counts)),
        'mean_target': float(np.mean(batch.guidance_targets)),
        'median_target': float(np.median(batch.guidance_targets)),
    }


def cycle_summary_frame(history_rows):
    rows = []
    for row in history_rows:
        bucket_counts = ', '.join(
            f"{bucket['label']}:{bucket['count']}"
            for bucket in row['cycle_bucket_summaries']
        )
        rows.append({
            'cycle': row['cycle'],
            'collected_examples': row['collected_examples'],
            'unguided_count': row['unguided_count'],
            'guided_count': row['guided_count'],
            'feasible_rate': row['cycle_feasible_rate'],
            'mean_violations': row['cycle_mean_violation'],
            'median_violations': row['cycle_median_violation'],
            'mean_target': row['cycle_mean_target'],
            'median_target': row['cycle_median_target'],
            'replay_buffer_size': row['replay_buffer_size'],
            'bucket_counts': bucket_counts,
            'train_skipped': not row['train_ran'],
            'skip_reason': row['train_skipped_reason'],
        })
    return pd.DataFrame(rows)


def bucket_plot_payload(graph_generator, batch, per_bucket=4, positive_bucket_count=8, random_state=42):
    rng = np.random.default_rng(random_state)
    payload = []
    for bucket in graph_generator.build_guidance_violation_buckets(
        batch.violation_counts,
        positive_bucket_count=positive_bucket_count,
    ):
        bucket_indices = np.asarray(bucket['indices'], dtype=int)
        if bucket_indices.size == 0:
            continue
        selected = rng.choice(
            bucket_indices,
            size=min(per_bucket, bucket_indices.size),
            replace=False,
        )
        selected = np.asarray(selected, dtype=int)
        payload.append({
            'label': bucket['label'],
            'count': int(bucket_indices.size),
            'median_violation': float(np.median(np.asarray(batch.violation_counts)[bucket_indices])),
            'median_target': float(np.median(np.asarray(batch.guidance_targets)[bucket_indices])),
            'graphs': [batch.decoded_graphs[int(i)] for i in selected],
            'legends': [
                f"v={int(batch.violation_counts[int(i)])}\nt={float(batch.guidance_targets[int(i)]):.3f}"
                for i in selected
            ],
        })
    return payload


def plot_cycle_bucket_samples(graph_generator, batch, cycle_row, per_bucket=4, positive_bucket_count=8, random_state=42):
    payload = bucket_plot_payload(
        graph_generator,
        batch,
        per_bucket=per_bucket,
        positive_bucket_count=positive_bucket_count,
        random_state=random_state,
    )
    if not payload:
        print(f"Cycle {cycle_row['cycle']}: no examples collected.")
        return
    title = (
        f"Cycle {cycle_row['cycle']} | unguided={cycle_row['unguided_count']} guided={cycle_row['guided_count']} | "
        f"collected={cycle_row['collected_examples']} | feasible_rate={cycle_row['cycle_feasible_rate']:.1%} | "
        f"mean_violations={cycle_row['cycle_mean_violation']:.2f} | train_ran={cycle_row['train_ran']}"
    )
    print(title)
    for bucket in payload:
        bucket_title = (
            f"{bucket['label']} | count={bucket['count']} | "
            f"median_violations={bucket['median_violation']:.2f} | median_target={bucket['median_target']:.3f}"
        )
        show_molecules(
            bucket['graphs'],
            title=bucket_title,
            legends=bucket['legends'],
            n_graphs_per_line=min(per_bucket, len(bucket['graphs'])),
        )


In [ ]:
# Build the cached ZINC subset once, then keep a debug-friendly train/test split.

ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
ZINC_MAX_MOLECULES = 250_000
MIN_NODE_COUNT = 10
MAX_NODE_COUNT = 20
TEST_SIZE = 256
RANDOM_STATE = 42
DEBUG_MODE = True
DEBUG_TRAIN_SUBSET = 6000
DEBUG_TEST_SUBSET = 64

csv_path = download_zinc_dataset(ZINC_DATA_ROOT)
corpus_manifest = build_zinc_graph_corpus(ZINC_DATA_ROOT, csv_path=csv_path)
graphs, zinc_metadata = load_zinc_graph_dataset(
    ZINC_DATA_ROOT,
    max_molecules=ZINC_MAX_MOLECULES,
    min_node_count=MIN_NODE_COUNT,
    max_node_count=MAX_NODE_COUNT,
)

print(f'ZINC CSV cache: {csv_path}')
print(f'ZINC graph cache root: {ZINC_DATA_ROOT}')
print(f'Node-count filter: {MIN_NODE_COUNT} <= n <= {MAX_NODE_COUNT}')
print(f"Available node counts: {corpus_manifest['node_counts'][:10]} ... {corpus_manifest['node_counts'][-10:]}")

target_columns = ['logP', 'qed', 'SAS']
targets = torch.tensor(
    extract_zinc_targets(zinc_metadata, target_columns=target_columns).to_numpy(dtype=np.float32),
    dtype=torch.float32,
)

summarize_graphs(graphs, prefix='zinc')
display(zinc_metadata.head())
display(pd.DataFrame(targets.numpy(), columns=target_columns).head())
show_molecules(graphs, n=12, title='Example ZINC molecules', n_graphs_per_line=4)

all_indices = np.arange(len(graphs))
effective_test_size = min(TEST_SIZE, max(1, len(all_indices) // 10))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=effective_test_size,
    random_state=RANDOM_STATE,
)
train_graphs = [graphs[int(i)] for i in train_indices]
test_graphs = [graphs[int(i)] for i in test_indices]
train_targets = targets[torch.as_tensor(train_indices, dtype=torch.long)]
test_targets = targets[torch.as_tensor(test_indices, dtype=torch.long)]

if DEBUG_MODE:
    rng = np.random.default_rng(RANDOM_STATE)
    train_pick = rng.choice(len(train_graphs), size=min(DEBUG_TRAIN_SUBSET, len(train_graphs)), replace=False)
    test_pick = rng.choice(len(test_graphs), size=min(DEBUG_TEST_SUBSET, len(test_graphs)), replace=False)
    train_graphs = [train_graphs[int(i)] for i in train_pick]
    test_graphs = [test_graphs[int(i)] for i in test_pick]
    train_targets = train_targets[torch.as_tensor(train_pick, dtype=torch.long)]
    test_targets = test_targets[torch.as_tensor(test_pick, dtype=torch.long)]

print(f'train_graphs={len(train_graphs)} test_graphs={len(test_graphs)}')
print(f'train_targets.shape={tuple(train_targets.shape)} test_targets.shape={tuple(test_targets.shape)}')
show_molecules(train_graphs, n=min(16, len(train_graphs)), title='Training subset preview', n_graphs_per_line=4)


In [ ]:
# Build the ZINC generator. If MODEL_FILENAME is set, the next cell will load it instead of fitting.

NBITS = 11
VERBOSE = 2
MODEL_NAME = f'demo-zinc-bootstrap-n{len(train_graphs)}-size{MIN_NODE_COUNT}-{MAX_NODE_COUNT}'
MODEL_FILENAME = None
FIT_IF_MISSING = True

graph_generator = build_graph_generator(
    nbits=NBITS,
    verbose=VERBOSE,
    feasibility_parallel=False,

    latent_embedding_dimension=128,
    number_of_transformer_layers=3,
    transformer_attention_head_count=4,
    transformer_dropout=0.15,

    learning_rate=2e-4,
    maximum_epochs=200,
    batch_size=16,
    total_steps=100,
    verbose_epoch_interval=10,
    enable_early_stopping=True,
    early_stopping_monitor='val_total',
    early_stopping_mode='min',
    early_stopping_patience=30,
    early_stopping_min_delta=10.0,
    restore_best_checkpoint=True,

    lambda_direct_edge_importance=2.0,
    lambda_auxiliary_edge_importance=1.0,
    lambda_degree_importance=2.0,
    lambda_degree_edge_consistency_importance=0.5,
    lambda_node_exist_importance=2.0,
    lambda_node_count_importance=0.5,
    lambda_node_label_importance=2.0,
    lambda_edge_label_importance=2.0,
    lambda_edge_count_importance=0.5,

    degree_temperature=1.0,
    node_field_sigma=0.2,
    sampling_step_size=0.05,
    langevin_noise_scale=0.0,

    locality_sample_fraction=0.5,
    locality_horizon=1,
    negative_sample_factor=1,
    locality_sampling_strategy='stratified_preserve',
    locality_target_positive_ratio=0.5,
    use_feasibility_filtering=True,
    max_feasibility_attempts=11,
    feasibility_candidates_per_attempt=512,
    feasibility_failure_mode='return_partial',
    feasibility_n_jobs=-1,
    decoder_n_jobs=-1,

    decoder_existence_threshold=0.5,
    decoder_enforce_connectivity=True,
    decoder_degree_slack_penalty=1e6,
    decoder_warm_start_mst=True,

    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
)


In [ ]:
%%time
# Load a saved full generator when MODEL_FILENAME is set; otherwise fit from scratch when FIT_IF_MISSING is True.

if MODEL_FILENAME:
    graph_generator = load_graph_generator(MODEL_FILENAME, model_dir=SAVED_GENERATOR_ROOT)
    print(f'Loaded full generator: {MODEL_FILENAME}')
elif FIT_IF_MISSING:
    graph_generator = fit_graph_generator(
        graph_generator,
        train_graphs,
        resume_latest_checkpoint=False,
        checkpoint_root=CHECKPOINT_ROOT,
    )
else:
    raise RuntimeError('Set MODEL_FILENAME or FIT_IF_MISSING=True before running bootstrap.')

print(f'model_name={getattr(graph_generator, "model_name", None)}')
print(f'model_dir={getattr(graph_generator, "model_dir", None)}')


In [ ]:
# Bootstrap configuration.

BOOTSTRAP_CYCLES = 4
EXAMPLES_PER_CYCLE = 96
INTERPOLATE_BETWEEN_N_SAMPLES = 10
REPLAY_TRAIN_SIZE = 256
POSITIVE_VIOLATION_BUCKETS = 8
GUIDED_FRACTION_AFTER_FIRST_CYCLE = 0.5
GUIDED_TARGET = 1.0
GUIDANCE_LEARNING_RATE = 1e-3
GUIDANCE_MAXIMUM_EPOCHS = 25
GUIDANCE_BATCH_SIZE = 64
GUIDANCE_NOISE_SCALE = 0.0
GUIDANCE_PREDICTOR_SCALE = 1.0
PLOT_MOLECULES_PER_BUCKET = 4
PLOT_RANDOM_STATE = 42

print('Guidance target formula: 1 / (1 + log1p(number_of_violations))')


In [ ]:
%%time
# Run the generated-sample bootstrap loop.

bootstrap_results = graph_generator.bootstrap_guidance_regressor_from_generated(
    num_cycles=BOOTSTRAP_CYCLES,
    examples_per_cycle=EXAMPLES_PER_CYCLE,
    interpolate_between_n_samples=INTERPOLATE_BETWEEN_N_SAMPLES,
    replay_train_size=REPLAY_TRAIN_SIZE,
    positive_bucket_count=POSITIVE_VIOLATION_BUCKETS,
    guided_fraction_after_first_cycle=GUIDED_FRACTION_AFTER_FIRST_CYCLE,
    guided_target=GUIDED_TARGET,
    guidance_learning_rate=GUIDANCE_LEARNING_RATE,
    guidance_maximum_epochs=GUIDANCE_MAXIMUM_EPOCHS,
    guidance_batch_size=GUIDANCE_BATCH_SIZE,
    guidance_noise_scale=GUIDANCE_NOISE_SCALE,
    predictor_scale=GUIDANCE_PREDICTOR_SCALE,
    random_state=PLOT_RANDOM_STATE,
)

cycle_history = bootstrap_results['history']
cycle_batches = bootstrap_results['cycle_batches']
replay_buffer = bootstrap_results['replay_buffer']
print(f'cycles={len(cycle_history)} replay_buffer={len(replay_buffer)}')


In [ ]:
# Compact cycle summary table.

summary_df = cycle_summary_frame(cycle_history)
display(summary_df)


In [ ]:
# Plot a random stratified sample from each non-empty violation bucket for every cycle.

for cycle_row, cycle_batch in zip(cycle_history, cycle_batches):
    plot_cycle_bucket_samples(
        graph_generator,
        cycle_batch,
        cycle_row,
        per_bucket=PLOT_MOLECULES_PER_BUCKET,
        positive_bucket_count=POSITIVE_VIOLATION_BUCKETS,
        random_state=PLOT_RANDOM_STATE + int(cycle_row['cycle']),
    )


In [ ]:
# Compare fresh unguided and regression-guided interpolation samples after bootstrap.

COMPARISON_SAMPLES = 24
unguided_batch = graph_generator.collect_generated_guidance_examples(
    n_samples=COMPARISON_SAMPLES,
    interpolate_between_n_samples=INTERPOLATE_BETWEEN_N_SAMPLES,
    sampling_mode='unguided',
)

comparison_rows = [summarize_generated_batch(unguided_batch, 'unguided')]
show_molecules(
    unguided_batch.decoded_graphs,
    n=COMPARISON_SAMPLES,
    title='Unguided interpolation samples',
    legends=[f"v={int(v)}\nt={float(t):.3f}" for v, t in zip(unguided_batch.violation_counts, unguided_batch.guidance_targets)],
    n_graphs_per_line=6,
)

guidance_ready = bool(
    getattr(graph_generator.conditional_node_generator_model, 'guidance_predictor_', None) is not None
    and getattr(graph_generator.conditional_node_generator_model, 'guidance_predictor_mode_', None) == 'regression'
)
if not guidance_ready:
    print('Regression guidance predictor is not available; skipping guided comparison samples.')
else:
    guided_batch = graph_generator.collect_generated_guidance_examples(
        n_samples=COMPARISON_SAMPLES,
        interpolate_between_n_samples=INTERPOLATE_BETWEEN_N_SAMPLES,
        sampling_mode='regression_guided',
        desired_target=GUIDED_TARGET,
        predictor_scale=GUIDANCE_PREDICTOR_SCALE,
    )
    comparison_rows.append(summarize_generated_batch(guided_batch, 'regression_guided'))
    show_molecules(
        guided_batch.decoded_graphs,
        n=COMPARISON_SAMPLES,
        title='Regression-guided interpolation samples',
        legends=[f"v={int(v)}\nt={float(t):.3f}" for v, t in zip(guided_batch.violation_counts, guided_batch.guidance_targets)],
        n_graphs_per_line=6,
    )

    compare_df = pd.DataFrame(comparison_rows)
    display(compare_df)

    plt.figure(figsize=(7, 4))
    plt.hist(unguided_batch.violation_counts, bins=20, alpha=0.6, label='unguided')
    plt.hist(guided_batch.violation_counts, bins=20, alpha=0.6, label='regression_guided')
    plt.xlabel('number_of_violations')
    plt.ylabel('count')
    plt.title('Violation-count distribution after bootstrap')
    plt.legend()
    plt.show()
